In [1]:
# ── Cell 0: Imports · Paths · Load v7 Config ──────────────────────────────────
import subprocess, re, math, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

# ══════════════════════════════════════════════════════════════════════════════
# HOUSE TYPE — change this one variable to switch between detached and semi
# ══════════════════════════════════════════════════════════════════════════════
HOUSE_TYPE = "semi"   # "detached"  or  "semi"

# ── Paths ──────────────────────────────────────────────────────────────────────
_root   = Path("/Users/hlbao/Library/CloudStorage/OneDrive-UniversityCollegeLondon/dissertation")
_proj   = _root / "BENV0166_2025"
_ep     = Path("/Applications/EnergyPlus-25-1-0")
_fm     = _root / "formal model file"

if HOUSE_TYPE == "semi":
    IDF_TEMPLATE = _fm / "semidetached_template.idf"
    CONFIG_FILE  = _fm / "parameter_config_v7_semi.json"
    OUT_DIR      = _fm / "output" / "pilot_v7_semi"
else:
    IDF_TEMPLATE = _fm / "detachedhouse0.idf"
    CONFIG_FILE  = _fm / "parameter_config_v7.json"
    OUT_DIR      = _fm / "output" / "pilot_v7"

WEATHER_FILE  = _proj / "weatherData" / "GBR_ENG_London.Wea.Ctr-St.James.Park.037700_TMYx.2009-2023.epw"
EP_EXE        = _ep   / "energyplus"

print(f"House type   : {HOUSE_TYPE}")
print(f"IDF template : {IDF_TEMPLATE.name}")
print(f"Config       : {CONFIG_FILE.name}")
print(f"Output dir   : {OUT_DIR}")

# ── Load JSON config ───────────────────────────────────────────────────────────
cfg = json.loads(CONFIG_FILE.read_text())
meta = cfg.get('metadata', cfg.get('_metadata', {}))

print(f"\nVersion  : {meta.get('version', '?')}")
print(f"Template : {meta.get('idf_template', '?')}")
print(f"LHS plan : {meta.get('lhs_samples', 2000)} samples (seed {meta.get('lhs_seed', 42)})")
print(f"\n{'Parameter':<30} {'Range':>18}  Unit")
print("─" * 55)
for p in cfg['sampled_parameters']:
    rng = f"[{p['min']}, {p['max']}]"
    print(f"  {p['name']:<28} {rng:>18}  {p.get('unit','')}")
if 'fixed_parameters' in cfg:
    print(f"\n{'Fixed':<30} {'Value':>18}")
    print("─" * 55)
    for k, v in cfg['fixed_parameters'].items():
        if isinstance(v, dict) and 'value' in v:
            print(f"  {k:<28} {str(v['value']):>18}  {v.get('unit','')}")

House type   : semi
IDF template : semidetached_template.idf
Config       : parameter_config_v7_semi.json
Output dir   : /Users/hlbao/Library/CloudStorage/OneDrive-UniversityCollegeLondon/dissertation/formal model file/output/pilot_v7_semi

Version  : v7-semi
Template : semidetached_template.idf
LHS plan : 2000 samples (seed 42)

Parameter                                   Range  Unit
───────────────────────────────────────────────────────
  width                               [6.0, 10.0]  m
  length                              [6.0, 12.0]  m
  height                               [2.4, 3.0]  m
  orientation                        [0.0, 360.0]  deg
  wwr                                  [0.1, 0.4]  —
  wallInsulationThickness             [0.01, 0.3]  m
  roofInsulationThickness           [0.025, 0.35]  m
  slabInsulationThickness             [0.01, 0.2]  m
  u_windows                            [0.8, 5.0]  W/m²K
  g_value                              [0.2, 0.7]  —
  roofAbsorptance   

In [2]:
# ── Cell 1: Pilot Parameters & All Derived Values ────────────────────────────

# ── Primary pilot values ──────────────────────────────────────────────────────
WIDTH        = 8.0    # m  (sampled: 6–10)
LENGTH       = 10.0   # m  (sampled: 6–12)
HEIGHT       = 2.7    # m  (sampled: 2.4–3)
ORIENTATION  = 180.0  # deg (0=N, 180=S facade faces south)
WWR          = 0.25   # window-to-wall ratio (sampled: 0.1–0.4)
WOF          = 0.30   # window opening fraction (sampled: 0.1–0.8)
WALL_INS     = 0.10   # m
ROOF_INS     = 0.20   # m
SLAB_INS     = 0.05   # m
U_WIN        = 2.0    # W/m²K
G_VALUE      = 0.4    # –
ROOF_ABS     = 0.7    # –
FLOW_COEFF   = 0.005  # m³/s  (v7 JSON range 0.001–0.012)
SHADING_DEPTH = 0.3   # m

# ── Fixed parameters (from v7 JSON) ──────────────────────────────────────────
HEATING_SP = 21.0    # °C daytime setpoint
COOLING_SP = 100.0   # °C  → free-running (no mechanical cooling)
VENT_RATE  = 0.0     # m³/s  → pure natural ventilation
INT_MASS   = 20.0    # m²   per zone

# ── Door collision constants ──────────────────────────────────────────────────
DOOR_START_X = 4.475   # hardcoded door on ground-floor south wall
DOOR_END_X   = 5.475
MARGIN       = 0.5     # wall-edge clearance
GAP          = 0.3     # gap between adjacent windows

# ── Window z-coordinates ──────────────────────────────────────────────────────
z1            = 0.8
z2            = min(2.2, HEIGHT - 0.3)
window_height = z2 - z1
print(f"Window height : {window_height:.3f} m  (z1={z1:.3f}, z2={z2:.3f})")

# ── S/N windows — WWR-based x-coordinates ────────────────────────────────────
# target area per face per floor = WWR × WIDTH × window_height
# split across 3 equal windows
single_w_SN = (WWR * WIDTH * window_height) / (3.0 * window_height)   # = WWR*WIDTH/3
print(f"S/N single window width : {single_w_SN:.4f} m")

ws_x1 = MARGIN
ws_x2 = ws_x1 + single_w_SN
ws_x3 = ws_x2 + GAP
ws_x4 = ws_x3 + single_w_SN
ws_x5 = ws_x4 + GAP
ws_x6 = ws_x5 + single_w_SN

# door collision check on window 3 (x5–x6)
if ws_x5 < DOOR_END_X and ws_x6 > DOOR_START_X:
    ws_x5 = DOOR_END_X + 0.05
    ws_x6 = ws_x5 + single_w_SN
    print(f"  ! Door collision → window 3 pushed right: x5={ws_x5:.4f}, x6={ws_x6:.4f}")

# clamp window 3 if it exceeds the wall
if ws_x6 > WIDTH - MARGIN:
    ws_x6 = WIDTH - MARGIN
    ws_x5 = max(DOOR_END_X + 0.05, ws_x6 - single_w_SN)
    print(f"  ! Window 3 clamped to wall: x5={ws_x5:.4f}, x6={ws_x6:.4f}")

print(f"S/N coords : [{ws_x1:.4f},{ws_x2:.4f}]  [{ws_x3:.4f},{ws_x4:.4f}]  [{ws_x5:.4f},{ws_x6:.4f}]")

# ── E/W window — WWR-based y-coordinates ─────────────────────────────────────
ew_w    = WWR * LENGTH                # 1 window per face
y_ctr   = LENGTH / 2.0
ew_y1   = max(0.5, y_ctr - ew_w / 2.0)
ew_y2   = min(LENGTH - 0.5, y_ctr + ew_w / 2.0)
print(f"E/W y-coords : [{ew_y1:.4f}, {ew_y2:.4f}]  (width={ew_y2-ew_y1:.4f} m)")

# ── Window opening areas : WOF × actual window area per face per floor ────────
actual_area_SN = 3 * single_w_SN * window_height
actual_area_EW = (ew_y2 - ew_y1) * window_height
open_S = WOF * actual_area_SN
open_N = WOF * actual_area_SN
open_E = WOF * actual_area_EW
open_W = WOF * actual_area_EW
print(f"\nOpening areas [m²] :  S={open_S:.5f}  N={open_N:.5f}  E={open_E:.5f}  W={open_W:.5f}")

# ── SAP 2012 occupancy ────────────────────────────────────────────────────────
TFA = 2 * WIDTH * LENGTH
N   = 1 + 1.76*(1 - math.exp(-0.000349*(TFA-13.9)**2)) + 0.0013*(TFA-13.9)
print(f"\nSAP 2012 : TFA={TFA:.1f} m²  →  N={N:.4f} people")

# ── Other derived parameters ──────────────────────────────────────────────────
two_height = 2 * HEIGHT
len_half   = LENGTH / 2.0
dl_ref_x   = WIDTH  / 2.0
dl_ref_y   = LENGTH / 2.0
print(f"\n2height={two_height}  length/2={len_half}  daylightRef=({dl_ref_x},{dl_ref_y})")

print("\n── All derived values ready for IDF substitution ──")


Window height : 1.400 m  (z1=0.800, z2=2.200)
S/N single window width : 0.6667 m
S/N coords : [0.5000,1.1667]  [1.4667,2.1333]  [2.4333,3.1000]
E/W y-coords : [3.7500, 6.2500]  (width=2.5000 m)

Opening areas [m²] :  S=0.84000  N=0.84000  E=1.05000  W=1.05000

SAP 2012 : TFA=160.0 m²  →  N=2.9489 people

2height=5.4  length/2=5.0  daylightRef=(4.0,5.0)

── All derived values ready for IDF substitution ──


In [3]:
# ── Cell 2: Build IDF & Run EnergyPlus ───────────────────────────────────────

# ── Substitution dictionary (placeholder → value) ────────────────────────────
subs = {
    "@width@"                    : f"{WIDTH}",
    "@length@"                   : f"{LENGTH}",
    "@height@"                   : f"{HEIGHT}",
    "@orientation@"              : f"{ORIENTATION}",
    "@wallInsulationThickness@"  : f"{WALL_INS}",
    "@roofInsulationThickness@"  : f"{ROOF_INS}",
    "@slabInsulationThickness@"  : f"{SLAB_INS}",
    "@u_windows@"                : f"{U_WIN}",
    "@g_value@"                  : f"{G_VALUE}",
    "@roofAbsorptance@"          : f"{ROOF_ABS}",
    "@flowCoefficient@"          : f"{FLOW_COEFF}",
    "@fixedShadingDepth@"        : f"{SHADING_DEPTH}",
    "@heatingSetpoint@"          : f"{HEATING_SP}",
    "@coolingSetpoint@"          : f"{COOLING_SP}",
    "@ventilationRate@"          : f"{VENT_RATE}",
    "@internalMass@"             : f"{INT_MASS}",
    "@numPeople@"                : f"{N:.6f}",
    "@2height@"                  : f"{two_height}",
    "@length/2@"                 : f"{len_half}",
    "@windowsouth_x1@"           : f"{ws_x1:.6f}",
    "@windowsouth_x2@"           : f"{ws_x2:.6f}",
    "@windowsouth_x3@"           : f"{ws_x3:.6f}",
    "@windowsouth_x4@"           : f"{ws_x4:.6f}",
    "@windowsouth_x5@"           : f"{ws_x5:.6f}",
    "@windowsouth_x6@"           : f"{ws_x6:.6f}",
    "@windowEast_y1@"            : f"{ew_y1:.6f}",
    "@windowEast_y2@"            : f"{ew_y2:.6f}",
    "@window_z1@"                : f"{z1:.6f}",
    "@window_z2@"                : f"{z2:.6f}",
    "@windowOpeningArea_S@"      : f"{open_S:.8f}",
    "@windowOpeningArea_N@"      : f"{open_N:.8f}",
    "@windowOpeningArea_E@"      : f"{open_E:.8f}",
    "@windowOpeningArea_W@"      : f"{open_W:.8f}",
    "@daylightReference_x@"      : f"{dl_ref_x}",
    "@daylightReference_y@"      : f"{dl_ref_y}",
}

# ── Apply substitutions ───────────────────────────────────────────────────────
template = IDF_TEMPLATE.read_text()
idf_text = template
for k, v in subs.items():
    idf_text = idf_text.replace(k, v)

# Check for unreplaced placeholders
remaining = re.findall(r'@[\w/]+@', idf_text)
if remaining:
    print(f"WARNING: {len(remaining)} unreplaced placeholders: {sorted(set(remaining))}")
else:
    print("All placeholders substituted successfully.")

# ── Write generated IDF ───────────────────────────────────────────────────────
OUT_DIR.mkdir(parents=True, exist_ok=True)
idf_out = OUT_DIR / "pilot.idf"
idf_out.write_text(idf_text)
print(f"IDF written  → {idf_out}")
print(f"Output dir   → {OUT_DIR}")

# ── Run EnergyPlus ───────────────────────────────────────────────────────────
ep_cmd = [
    str(EP_EXE),
    "-w", str(WEATHER_FILE),
    "-d", str(OUT_DIR),
    "-r",                      # readvars: produce eplusout.csv
    str(idf_out),
]
print(f"\nRunning EnergyPlus 25.1 …")
result = subprocess.run(ep_cmd, capture_output=True, text=True, timeout=600)

if result.returncode == 0:
    print("EnergyPlus completed  [exit 0]")
else:
    print(f"EnergyPlus exit code: {result.returncode}")
    lines = (result.stdout + result.stderr).splitlines()
    print("\n── EnergyPlus stdout/stderr (last 60 lines) ──")
    print('\n'.join(lines[-60:]))

# Parse .err file — EnergyPlus marks real errors as "** Severe  **" / "** Fatal  **"
err_file = OUT_DIR / "eplusout.err"
if err_file.exists():
    err_lines = err_file.read_text().splitlines()
    severe   = [l for l in err_lines if '** Severe' in l or '** Fatal' in l]
    warnings = [l for l in err_lines if '** Warning' in l]
    if severe:
        print(f"\n── Severe/Fatal errors ({len(severe)}) ──")
        print('\n'.join(severe[:20]))
    else:
        print(f"Error file: 0 severe/fatal errors, {len(warnings)} warnings — simulation clean")


All placeholders substituted successfully.
IDF written  → /Users/hlbao/Library/CloudStorage/OneDrive-UniversityCollegeLondon/dissertation/formal model file/output/pilot_v7_semi/pilot.idf
Output dir   → /Users/hlbao/Library/CloudStorage/OneDrive-UniversityCollegeLondon/dissertation/formal model file/output/pilot_v7_semi

Running EnergyPlus 25.1 …
EnergyPlus completed  [exit 0]
Error file: 0 severe/fatal errors, 8 warnings — simulation clean


In [4]:
# ── Cell 3: Parse EnergyPlus CSV Output ──────────────────────────────────────
import matplotlib
matplotlib.use('Agg')          # non-interactive backend for notebooks
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta

csv_path = OUT_DIR / "eplusout.csv"
if not csv_path.exists():
    raise FileNotFoundError(
        f"No CSV at {csv_path}\n"
        f"Check error file: {OUT_DIR / 'eplusout.err'}"
    )

df_raw = pd.read_csv(csv_path)
print(f"Loaded  {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")
print("\nAll columns:")
for c in df_raw.columns:
    print(f"  {c}")

# ── Flexible column finder (case-insensitive keyword search) ──────────────────
def find_col(df, *keywords, exclude=None):
    """Return first column whose name contains ALL keywords (case-insensitive)."""
    kws = [k.lower() for k in keywords]
    for col in df.columns:
        low = col.lower()
        if all(k in low for k in kws):
            if exclude is None or exclude.lower() not in low:
                return col
    return None

# Outdoor drybulb
col_oat   = find_col(df_raw, 'outdoor', 'drybulb')
# Operative temperatures per zone
col_op101 = find_col(df_raw, '101', 'operative')
col_op102 = find_col(df_raw, '102', 'operative')
# Air temperatures (fallback)
col_at101 = find_col(df_raw, '101', 'air temperature')
col_at102 = find_col(df_raw, '102', 'air temperature')
# Heating & cooling energy (J → kWh)
col_heat101 = find_col(df_raw, '101', 'heating energy')
col_heat102 = find_col(df_raw, '102', 'heating energy')
col_cool101 = find_col(df_raw, '101', 'cooling energy')
col_cool102 = find_col(df_raw, '102', 'cooling energy')

print(f"\nKey columns identified:")
print(f"  Outdoor drybulb    : {col_oat}")
print(f"  Operative temp 101 : {col_op101}  (fallback air: {col_at101})")
print(f"  Operative temp 102 : {col_op102}  (fallback air: {col_at102})")
print(f"  Heating energy 101 : {col_heat101}")
print(f"  Heating energy 102 : {col_heat102}")

# ── Build clean hourly DataFrame ──────────────────────────────────────────────
# EnergyPlus Date/Time: " 1/ 1 01:00:00" … "12/31 24:00:00"  (8760 rows)
n_rows = len(df_raw)
base = datetime(2023, 1, 1)    # reference year (matches TMYx 2009-2023 EPW)
timestamps = [base + timedelta(hours=h) for h in range(n_rows)]

df = pd.DataFrame({'timestamp': timestamps})
df['doy']   = df['timestamp'].dt.dayofyear - 1   # 0-based (0=Jan 1)
df['hour']  = df['timestamp'].dt.hour
df['month'] = df['timestamp'].dt.month

df['T_out'] = df_raw[col_oat].values if col_oat else np.nan

# Prefer operative temperature; fall back to air temperature
use_op101 = col_op101 is not None
use_op102 = col_op102 is not None
df['T_op101'] = df_raw[col_op101].values if use_op101 else df_raw[col_at101].values
df['T_op102'] = df_raw[col_op102].values if use_op102 else df_raw[col_at102].values
df['T_op_avg'] = (df['T_op101'] + df['T_op102']) / 2.0

# Energy (J → kWh, NaN if column missing)
J2kWh = 1.0 / 3_600_000.0
def safe_kWh(col_name):
    return df_raw[col_name].values * J2kWh if col_name else np.full(n_rows, np.nan)

df['Q_heat101'] = safe_kWh(col_heat101)
df['Q_heat102'] = safe_kWh(col_heat102)
df['Q_cool101'] = safe_kWh(col_cool101)
df['Q_cool102'] = safe_kWh(col_cool102)

temp_label = 'operative' if (use_op101 or use_op102) else 'air (no operative col found)'
print(f"\nUsing {temp_label} temperature for TM52 assessment")
print(df[['timestamp','T_out','T_op101','T_op102']].head(6).to_string(index=False))


Loaded  8760 rows × 37 columns

All columns:
  Date/Time
  Environment:Site Outdoor Air Drybulb Temperature [C](Hourly)
  LIGHTS_SPACE101:Lights Electricity Energy [J](Hourly)
  LIGHTS_SPACE102:Lights Electricity Energy [J](Hourly)
  ELECTRICEQUIPMENT_SPACE101:Electric Equipment Electricity Energy [J](Hourly)
  ELECTRICEQUIPMENT_SPACE102:Electric Equipment Electricity Energy [J](Hourly)
  THERMAL ZONE: SPACE 1:Zone Windows Total Heat Gain Rate [W](Hourly)
  THERMAL ZONE: SPACE 101:Zone Windows Total Heat Gain Rate [W](Hourly)
  THERMAL ZONE: SPACE 102:Zone Windows Total Heat Gain Rate [W](Hourly)
  THERMAL ZONE: SPACE 1:Zone Operative Temperature [C](Hourly)
  THERMAL ZONE: SPACE 101:Zone Operative Temperature [C](Hourly)
  THERMAL ZONE: SPACE 102:Zone Operative Temperature [C](Hourly)
  THERMAL ZONE: SPACE 1:Zone Infiltration Current Density Volume Flow Rate [m3/s](Hourly)
  THERMAL ZONE: SPACE 101:Zone Infiltration Current Density Volume Flow Rate [m3/s](Hourly)
  THERMAL ZONE: SPACE

In [5]:
# ── Cell 4: TM52 / TM59 Overheating Assessment + Energy Summary ──────────────
# All zone metrics computed SEPARATELY then worst-zone taken (TM52 standard).
# Continuous metrics used as surrogate model target variables.

# ── 1. Daily mean outdoor temperature ────────────────────────────────────────
daily_T_out = df.groupby('doy')['T_out'].mean().values
n_days = len(daily_T_out)

# ── 2. Exponentially weighted running mean Trm (α = 0.8) ─────────────────────
alpha = 0.8
Trm_arr = np.zeros(n_days + 1)
Trm_arr[0] = daily_T_out[0]
for d in range(n_days):
    Trm_arr[d+1] = (1 - alpha) * daily_T_out[d] + alpha * Trm_arr[d]
Trm_daily = Trm_arr[1:]

df['Trm']  = Trm_daily[df['doy'].values]
df['Tmax'] = 0.33 * df['Trm'] + 21.8    # CIBSE TM52 comfort limit
df['Tupp'] = df['Tmax'] + 4.0

# ── 3. Assessment period: May–Sep ────────────────────────────────────────────
summer_mask = df['month'].between(5, 9)
night_mask  = ~df['hour'].between(7, 21)   # 22:00–06:00 (TM59 bedroom hours)

df_summer = df[summer_mask].copy()
n_occ = len(df_summer)

# ── Per-zone TM52 calculation ─────────────────────────────────────────────────
def tm52_zone(T_op_series, Tmax_series, Tupp_series, doy_series, n_occ):
    """Returns (He_pct, We_max, exceed_tupp) for one zone."""
    dT   = T_op_series.values - Tmax_series.values
    # Criterion 1
    He   = 100.0 * (dT > 1.0).sum() / n_occ
    # Criterion 2 — stepped weight
    W    = np.where(dT <= 0, 0, np.where(dT <= 1, 1, np.where(dT <= 2, 2, 3)))
    We_d = pd.Series(W, index=doy_series.values).groupby(level=0).sum()
    We_max = We_d.max()
    worst_day = We_d.idxmax()
    # Criterion 3
    exc_tupp = (T_op_series.values > Tupp_series.values).sum()
    return He, We_max, exc_tupp, worst_day

He_101, We_101, exc3_101, wd_101 = tm52_zone(
    df_summer['T_op101'], df_summer['Tmax'], df_summer['Tupp'], df_summer['doy'], n_occ)
He_102, We_102, exc3_102, wd_102 = tm52_zone(
    df_summer['T_op102'], df_summer['Tmax'], df_summer['Tupp'], df_summer['doy'], n_occ)

# ── Worst-zone metrics (TM52 standard) ───────────────────────────────────────
He_worst      = max(He_101, He_102)
We_max_worst  = max(We_101, We_102)
exc3_worst    = max(exc3_101, exc3_102)

crit1_pass = He_worst  < 3.0
crit2_pass = We_max_worst <= 6
crit3_pass = exc3_worst == 0
all_pass   = crit1_pass and crit2_pass and crit3_pass

print("── CIBSE TM52 / TM59 Assessment (May–Sep, per-zone worst-case) ──────────")
print(f"\n                   GF (101)     FF (102)    WORST-ZONE")
print(f"  He [%]       :  {He_101:>8.2f}     {He_102:>8.2f}    {He_worst:>8.2f}   (< 3%)")
print(f"  We_max       :  {We_101:>8.0f}     {We_102:>8.0f}    {We_max_worst:>8.0f}   (≤ 6)")
print(f"  Tupp hours   :  {exc3_101:>8.0f}     {exc3_102:>8.0f}    {exc3_worst:>8.0f}   (= 0)")
print(f"\n  Crit 1 He  < 3%  → {'PASS ✓' if crit1_pass else 'FAIL ✗'}")
print(f"  Crit 2 We  ≤ 6   → {'PASS ✓' if crit2_pass else 'FAIL ✗'}")
print(f"  Crit 3 Tupp = 0  → {'PASS ✓' if crit3_pass else 'FAIL ✗'}")
print(f"\n{'='*50}")
print(f"  Overall TM52/TM59 : {'NO OVERHEATING ✓' if all_pass else 'OVERHEATING ✗'}")
print(f"{'='*50}")

# ── Surrogate model target variables ─────────────────────────────────────────
T_op_peak    = max(df_summer['T_op101'].max(), df_summer['T_op102'].max())

# TM59 bedroom: nights (22:00–06:00) in summer with T_op > 26°C
night_summer  = df_summer[night_mask[summer_mask]]
hours_gt26_night = int(
    (night_summer['T_op101'] > 26).sum() + (night_summer['T_op102'] > 26).sum()
) // 2   # average zones (both represent habitable rooms)

# Annual heating energy
total_heat   = df['Q_heat101'].sum() + df['Q_heat102'].sum()
floor_area   = 2 * WIDTH * LENGTH
heat_kWh_m2  = total_heat / floor_area

print(f"\n── Surrogate Model Target Variables ────────────────────────────────────")
print(f"  He_worst       = {He_worst:.4f}  %       ← primary overheating severity")
print(f"  We_max_worst   = {We_max_worst:.1f}              ← peak single-day exceedance")
print(f"  T_op_peak      = {T_op_peak:.2f}  °C      ← maximum operative temperature (summer)")
print(f"  hours_gt26_ngt = {hours_gt26_night:>5d}  h        ← TM59 bedroom: nights T>26°C")
print(f"  heat_kWh_m2    = {heat_kWh_m2:.2f}  kWh/m²·yr ← heating energy intensity")

# ── Store for Cell 5 ─────────────────────────────────────────────────────────
targets = dict(
    He_worst      = He_worst,
    We_max_worst  = We_max_worst,
    T_op_peak     = T_op_peak,
    hours_gt26_night = hours_gt26_night,
    heat_kWh_m2   = heat_kWh_m2,
)


── CIBSE TM52 / TM59 Assessment (May–Sep, per-zone worst-case) ──────────

                   GF (101)     FF (102)    WORST-ZONE
  He [%]       :      0.00         0.00        0.00   (< 3%)
  We_max       :         0            3           3   (≤ 6)
  Tupp hours   :         0            0           0   (= 0)

  Crit 1 He  < 3%  → PASS ✓
  Crit 2 We  ≤ 6   → PASS ✓
  Crit 3 Tupp = 0  → PASS ✓

  Overall TM52/TM59 : NO OVERHEATING ✓

── Surrogate Model Target Variables ────────────────────────────────────
  He_worst       = 0.0000  %       ← primary overheating severity
  We_max_worst   = 3.0              ← peak single-day exceedance
  T_op_peak      = 28.95  °C      ← maximum operative temperature (summer)
  hours_gt26_ngt =     1  h        ← TM59 bedroom: nights T>26°C
  heat_kWh_m2    = 19.32  kWh/m²·yr ← heating energy intensity


In [7]:
# ── Cell 5: Summer Temperature & Comfort Plot ────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
fig.suptitle(
    f"Pilot Simulation — W={WIDTH}m  L={LENGTH}m  H={HEIGHT}m  "
    f"Orientation={ORIENTATION}°  WWR={WWR}  WOF={WOF}\n"
    f"Wall={WALL_INS*100:.0f}mm  Roof={ROOF_INS*100:.0f}mm  "
    f"U_win={U_WIN} W/m²K  g={G_VALUE}  FlowCoeff={FLOW_COEFF} m³/s",
    fontsize=10
)

summer = df_summer.copy()
ts = summer['timestamp']

# ── Panel 1: Per-zone operative vs outdoor vs comfort limit ───────────────────
ax1 = axes[0]
ax1.plot(ts, summer['T_out'],    color='steelblue',  lw=1,         label='Outdoor drybulb')
ax1.plot(ts, summer['T_op101'],  color='tomato',     lw=1,         label='T_op GF (101)')
ax1.plot(ts, summer['T_op102'],  color='darkorange', lw=1,         label='T_op FF (102)')
ax1.plot(ts, summer['Tmax'],     color='black',      lw=1.5, ls='--', label='Tmax comfort limit')
ax1.fill_between(ts, summer['Tmax'], summer['Tupp'],
                 alpha=0.12, color='orange', label='Tmax–Tupp (+4K)')
ax1.axhline(26.0, color='purple', lw=0.8, ls=':', alpha=0.7, label='26°C (TM59 night threshold)')
ax1.set_ylabel('Temperature (°C)')
ax1.set_title('Operative Temperatures vs CIBSE TM52 Adaptive Comfort Limit  (May–Sep)')
ax1.legend(fontsize=8, ncol=3, loc='upper left')
ax1.grid(alpha=0.3)

# ── Panel 2: Per-zone ΔT exceedance ──────────────────────────────────────────
ax2 = axes[1]
dT_101 = (summer['T_op101'] - summer['Tmax']).clip(lower=0)
dT_102 = (summer['T_op102'] - summer['Tmax']).clip(lower=0)
ax2.fill_between(ts, dT_101, alpha=0.65, color='tomato',    label=f'ΔT GF  He={He_101:.2f}%')
ax2.fill_between(ts, dT_102, alpha=0.55, color='darkorange', label=f'ΔT FF  He={He_102:.2f}%')
ax2.axhline(1.0, color='black', ls='--', lw=1, label='ΔT = 1K  (Crit.1 threshold)')
ax2.set_ylabel('ΔT above Tmax (K)')
ax2.set_title(f'Hourly Exceedance — worst-zone He={He_worst:.2f}%  We_max={We_max_worst:.0f}  '
              f'T_op_peak={T_op_peak:.1f}°C')
ax2.legend(fontsize=8, loc='upper right')
ax2.grid(alpha=0.3)

# ── Panel 3: Running mean Trm ────────────────────────────────────────────────
ax3 = axes[2]
ax3.plot(ts, summer['T_out'], color='steelblue', lw=0.6, alpha=0.4, label='Hourly outdoor')
ax3.plot(ts, summer['Trm'],   color='navy',      lw=2,              label='Running mean Trm (α=0.8)')
ax3.plot(ts, summer['Tmax'],  color='black',     lw=1.5, ls='--',   label='Tmax = 0.33×Trm + 21.8')
ax3.set_ylabel('Temperature (°C)')
ax3.set_title('Exponentially Weighted Running Mean Outdoor Temperature')
ax3.legend(fontsize=8, loc='upper left')
ax3.grid(alpha=0.3)
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
ax3.xaxis.set_major_locator(mdates.MonthLocator())
fig.autofmt_xdate(rotation=0, ha='center')

# ── Surrogate targets annotation ─────────────────────────────────────────────
verdict   = 'PASS' if all_pass else 'FAIL'
box_color = 'honeydew' if all_pass else 'mistyrose'
annot = (
    f"TM52/TM59 [{verdict}]\n"
    f"He_worst     = {He_worst:.2f}%  ({'<3%✓' if crit1_pass else '≥3%✗'})\n"
    f"We_max_worst = {We_max_worst:.0f}  ({'≤6✓' if crit2_pass else '>6✗'})\n"
    f"Tupp hours   = {exc3_worst}  ({'=0✓' if crit3_pass else '>0✗'})\n"
    f"─────────────────\n"
    f"T_op_peak    = {T_op_peak:.1f}°C\n"
    f"Ngt>26°C     = {hours_gt26_night}h\n"
    f"Heating      = {heat_kWh_m2:.1f} kWh/m²"
)
ax1.text(0.99, 0.97, annot, transform=ax1.transAxes, fontsize=8,
         verticalalignment='top', horizontalalignment='right', family='monospace',
         bbox=dict(boxstyle='round,pad=0.4', facecolor=box_color, alpha=0.92))

plt.tight_layout()
fig_path = OUT_DIR / "pilot_v7_results.png"
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Figure saved → {fig_path}")

Figure saved → /Users/hlbao/Library/CloudStorage/OneDrive-UniversityCollegeLondon/dissertation/formal model file/output/pilot_v7_semi/pilot_v7_results.png
